# Experiment 10 — Buyer-template sensitivity

## Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import negolin_utils as nu
nu.setup_matplotlib()

# Override paths if your data lives elsewhere:
# nu.RAND_BUYER_DIR = Path('/abs/path/to/0421_redo_previous_exp/results')

# Load data and apply v7 preprocessing
raw = nu.load_all_fixed_policies(nu.RAND_BUYER_DIR)
_, turns = nu.preprocess_turns(raw)
POLICIES = list(nu.POLICIES)

# Load Stage-2 linear model selection from Exp02 (if present)
try:
    selected = pd.read_csv('selected_linear_models.csv')
    SELECTED = {row['policy']: (row['feature'], row['model'], row.get('alpha', np.nan))
                for _, row in selected.iterrows()}
    print('Loaded per-policy (feature, model, alpha) from Exp02.')
except FileNotFoundError:
    # Fall back to a reasonable default (Feature II, Ridge) if Exp02 hasn't run
    SELECTED = {p: ('II', 'Ridge', 1.0) for p in POLICIES}
    print('selected_linear_models.csv not found; using default (Feature II, Ridge).')

**Goal.** Verify the seller's policy is insensitive to the buyer's
paraphrase (the 8 message templates). Fit the policy model with and without
template dummies and compare MAE.

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge

rows = []
for p in POLICIES:
    feat, mdl, alpha = SELECTED[p]
    meta, X, y, cols = nu.build_design_matrix(turns, p, feat)
    templ = meta['buyer_template_id'].to_numpy().reshape(-1, 1) if 'buyer_template_id' in meta.columns else None
    # Base MAE (no template)
    pred_base, _ = nu.grouped_cv_predict(meta, X, y, mdl,
                                          n_splits=nu.OUTER_FOLDS,
                                          fixed_alpha=float(alpha) if (mdl == 'Ridge' and np.isfinite(alpha)) else None)
    mae_base = nu.cv_metrics_full(pred_base)['mae_norm']
    rows.append({'policy': p, 'variant': 'no_template', 'mae_norm': mae_base})

    if templ is not None and len(np.unique(templ)) > 1:
        enc = OneHotEncoder(drop='first', sparse_output=False).fit(templ)
        X_ext = np.hstack([X, enc.transform(templ)])
        pred_ext, _ = nu.grouped_cv_predict(meta, X_ext, y, mdl,
                                             n_splits=nu.OUTER_FOLDS,
                                             fixed_alpha=float(alpha) if (mdl == 'Ridge' and np.isfinite(alpha)) else None)
        mae_ext = nu.cv_metrics_full(pred_ext)['mae_norm']
        rows.append({'policy': p, 'variant': 'with_template', 'mae_norm': mae_ext})

template_tbl = pd.DataFrame(rows).pivot(index='policy', columns='variant', values='mae_norm')
template_tbl['delta'] = template_tbl['with_template'] - template_tbl['no_template']
template_tbl.round(5)